# Day 29 — 가설검정, 회귀, 분류: 질문에 맞는 분석 고르기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-5/day-29.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-5/day-29.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 준비 코드: 가상 feature table 만들고 구조 세어 보기

아래 "준비 코드"는 이번 주 내내 각 레슨의 첫 셀로 다시 등장합니다. seed를 고정했기 때문에 누가 돌려도 같은 표가 나옵니다.

In [ ]:
import numpy as np
import pandas as pd

FEATURES = ["stride_time_cv", "accel_rms", "tremor_band_power",
            "jerk_mean", "step_regularity", "arm_swing_asym"]

def make_demo_table(seed=42, n_participants=50, n_repeats=6):
    """연습용 가상 feature table. 실제 환자 데이터가 아니다."""
    rng = np.random.default_rng(seed)
    pd_shift = np.array([0.6, -0.5, 0.6, 0.2, -0.4, 0.3])   # PD군 평균이 이동하는 정도
    rows = []
    for i in range(n_participants):
        group = "PD" if i < n_participants // 2 else "HC"
        age = rng.normal(68 if group == "PD" else 62, 7)
        sex = rng.choice(["F", "M"])
        person = rng.normal(0, 1.0, size=len(FEATURES))      # 그 사람 고유의 '움직임 지문'
        for rep in range(1, n_repeats + 1):
            values = person + rng.normal(0, 0.25, size=len(FEATURES))  # 측정마다 생기는 흔들림
            if group == "PD":
                values = values + pd_shift
            values[0] += 0.06 * (age - 65)                   # stride_time_cv는 나이와도 관련
            rows.append({"participant_id": f"P{i + 1:03d}", "group": group,
                         "age": round(age, 1), "sex": sex, "repeat": rep,
                         **dict(zip(FEATURES, values.round(3)))})
    return pd.DataFrame(rows)

df = make_demo_table()

# 🔁 실제 데이터로 바꿀 지점 ---------------------------------------------
# 4주차(Day 26)에 만든 feature table이 있으면 위의 df = make_demo_table() 대신:
# df = pd.read_csv("data/physionet_pd/derived/pd_features_records.csv")   # 내 파일 경로로
# FEATURES = ["내_feature_1", "내_feature_2"]                # 내 feature 열 이름으로
# 그리고 아래 두 이름을 내 표의 열 이름에 맞춘다.
ID_COL = "participant_id"   # 참가자 ID 열
GROUP_COL = "group"         # 진단군 열 (이 예시에서는 "PD" / "HC")
# -------------------------------------------------------------------------

print("행(측정) 수:", len(df))
print("참가자 수:", df[ID_COL].nunique())
print("참가자당 측정 수:", df.groupby(ID_COL).size().unique())
print()
print("참가자 단위 군별 인원:")
print(df.drop_duplicates(ID_COL)[GROUP_COL].value_counts())
print()
print(df.head(3).to_string(index=False))

### 예시 2 — 질문 A(군 비교)와 질문 B(회귀): 답의 모양이 다르다

예시 1과 같은 노트북에서 이어서 실행합니다. 먼저 참가자별 평균으로 표를 줄인 다음, 두 가지 질문을 던집니다.

In [ ]:
from scipy import stats

# 참가자 단위 표: 한 사람 = 한 행
person = df.groupby([ID_COL, GROUP_COL], as_index=False)[["age"] + FEATURES].mean()

# 질문 A (군 비교): PD군과 HC군의 tremor_band_power 평균이 다른가?
pd_vals = person.loc[person[GROUP_COL] == "PD", "tremor_band_power"]
hc_vals = person.loc[person[GROUP_COL] == "HC", "tremor_band_power"]
t = stats.ttest_ind(pd_vals, hc_vals, equal_var=False)   # Welch t-검정
print("질문 A — 군 비교")
print(f"  평균 차이(PD - HC) = {pd_vals.mean() - hc_vals.mean():.2f},  p = {t.pvalue:.3f}")

# 질문 B (회귀): 나이가 1살 많으면 stride_time_cv가 얼마나 달라지는가?
reg = stats.linregress(person["age"], person["stride_time_cv"])
print("질문 B — 회귀")
print(f"  기울기 = {reg.slope:.3f} (나이 1살당),  r = {reg.rvalue:.2f}")

### 예시 3 — 질문 C(분류) 맛보기: 답은 "처음 보는 참가자를 맞힌 비율"

분류는 Day 32에서 차근차근 배웁니다. 오늘은 "답의 모양"만 봅니다. 참가자 50명 중 35명으로 규칙을 배우고, **나머지 15명**의 측정을 얼마나 맞히는지 셉니다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

X = df[FEATURES]
y = (df[GROUP_COL] == "PD").astype(int)      # PD = 1, HC = 0

# 참가자 단위로 학습용/평가용을 나눈다 (같은 사람은 한쪽에만)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
train_idx, test_idx = next(splitter.split(X, y, groups=df[ID_COL]))

model = LogisticRegression().fit(X.iloc[train_idx], y.iloc[train_idx])
acc = model.score(X.iloc[test_idx], y.iloc[test_idx])
print("질문 C — 분류")
print(f"  평가용 참가자 수 = {df[ID_COL].iloc[test_idx].nunique()}명")
print(f"  처음 보는 참가자의 측정을 맞힌 비율(accuracy) = {acc:.2f}")